# Week 11, Day 1: Sines, cosines and complex exponentials: describing rhythms

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w011/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Make and measure sinusoids

Write `sinusoid(t, f, A, phi)`. Sample one second of a 10 Hz sinusoid at 1 kHz and count its peaks (samples larger than both neighbours) in `n_peaks`. Then compute how many milliseconds earlier the peaks of a 10 Hz wave occur when its phase is π/2 (`shift_ms`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoid(t, f, A, phi):
    return ...

t = np.arange(0, 1, 0.001)
x = sinusoid(t, 10.0, 1.0, -np.pi / 2)            # a sine, so no peak falls on the first sample
n_peaks = int(np.sum((x[1:-1] > x[:-2]) & (x[1:-1] > x[2:])))
shift_ms = ...

plt.figure(figsize=(6, 2.5)); plt.plot(t[:300] * 1000, x[:300], label="phase 0"); plt.plot(t[:300] * 1000, sinusoid(t, 10, 1, np.pi / 2)[:300], label="phase π/2")
plt.xlabel("time (ms)"); plt.legend(frameon=False); plt.show()
print(n_peaks, "peaks;  shift", shift_ms, "ms")

In [ ]:
import numpy as np
def test_sine():
    "sinusoid returns A cos(2 pi f t + phi)"
    t = np.linspace(0, 1, 11)
    assert np.allclose(sinusoid(t, 3.0, 2.0, 0.5), 2.0 * np.cos(2 * np.pi * 3.0 * t + 0.5))

def test_peaks():
    "a 10 Hz sinusoid has 10 peaks per second"
    assert n_peaks == 10

def test_shift():
    "a phase of pi/2 at 10 Hz moves the peaks 25 ms earlier"
    assert np.isclose(shift_ms, 25.0)

test_sine()
test_peaks()
test_shift()
print("All checks passed")

In [ ]:
theta = np.linspace(0, 2 * np.pi, 9)
z = np.exp(1j * theta)
print("e^(i theta) for 9 angles:", np.round(z, 3))
print("lengths all 1:", np.allclose(np.abs(z), 1), "  angles:", np.round(np.angle(z), 3))
a, b = 2 * np.exp(1j * 0.3), 1.5 * np.exp(1j * 1.1)
print("product: length", round(abs(a * b), 3), "= 2 x 1.5;  angle", round(np.angle(a * b), 3), "= 0.3 + 1.1")

### Exercise: Convert between the two forms

Write `amp_phase(a, b)` returning the amplitude and phase of $a\cos\omega t + b\sin\omega t$. Check by plotting both forms for a = −1, b = 1.

In [ ]:
def amp_phase(a, b):
    return ...

A, phi = amp_phase(-1.0, 1.0)
t = np.linspace(0, 1, 200)
plt.figure(figsize=(5, 2.5)); plt.plot(t, -np.cos(5 * t) + np.sin(5 * t), lw=3, alpha=0.5, label="a cos + b sin"); plt.plot(t, A * np.cos(5 * t - phi), "--", label="A cos(ωt − φ)")
plt.legend(frameon=False); plt.show()
print("A =", round(A, 4), " phi =", round(phi, 4), "rad")

In [ ]:
import numpy as np
def test_values():
    "amp_phase returns the amplitude and phase"
    A, phi = amp_phase(3.0, 4.0)
    assert np.isclose(A, 5.0) and np.isclose(phi, np.arctan2(4, 3))

def test_quadrant():
    "it handles negative cosine coefficients correctly"
    A, phi = amp_phase(-1.0, 1.0)
    t = np.linspace(0, 1, 50)
    assert np.allclose(-np.cos(5 * t) + np.sin(5 * t), A * np.cos(5 * t - phi))

test_values()
test_quadrant()
print("All checks passed")

### Exercise: Find the alpha rhythm in the noise

The signal is a 10 Hz rhythm of amplitude 1 and phase 0.8 rad (as $A\cos(\omega t - \phi)$) plus noise of standard deviation 5, sampled at 500 Hz for 8 s. Project onto the cosine and sine at 10 Hz to get `a_hat` and `b_hat`, convert to `A_hat` and `phi_hat`, and compare. How large is the expected error, $5\sqrt{2/N}$?

In [ ]:
rng = np.random.default_rng(1929)
fs = 500
t2 = np.arange(0, 8, 1 / fs)
x2 = 1.0 * np.cos(2 * np.pi * 10 * t2 - 0.8) + 5 * rng.normal(size=len(t2))

a_hat = ...
b_hat = ...
A_hat, phi_hat = amp_phase(a_hat, b_hat)

plt.figure(figsize=(6, 2.5)); plt.plot(t2, x2, color="0.6", lw=0.6); plt.plot(t2, A_hat * np.cos(2 * np.pi * 10 * t2 - phi_hat), lw=2)
plt.xlim(0, 0.5); plt.xlabel("time (s)"); plt.ylabel("µV"); plt.show()
print(f"amplitude {A_hat:.3f} (true 1.0), phase {phi_hat:.3f} (true 0.8);  expected error about {5 * np.sqrt(2 / len(t2)):.3f}")

In [ ]:
import numpy as np
def test_coeffs():
    "a_hat and b_hat are the projections of the signal on cos and sin at 10 Hz"
    c, s = np.cos(2 * np.pi * 10 * t2), np.sin(2 * np.pi * 10 * t2)
    assert np.isclose(a_hat, 2 / len(t2) * x2 @ c) and np.isclose(b_hat, 2 / len(t2) * x2 @ s)

def test_amp():
    "the estimated amplitude is within 20% of the true 1 microvolt"
    assert abs(A_hat - 1.0) < 0.2

def test_phase():
    "the estimated phase is within 0.3 rad of the true 0.8 rad"
    assert abs(np.angle(np.exp(1j * (phi_hat - 0.8)))) < 0.3

test_coeffs()
test_amp()
test_phase()
print("All checks passed")